# Part 4: Evaluation

### Task 1 - Evaluate the performance of your Simple and Advanced Model on FakeNewsCorpus.

In [1]:
#Loading packages
import pickle
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC 

In [ ]:
#Loading dataset(s):
df = pd.read_csv("995,000_rows.csv", usecols=["type", "content"])
df["label"] = (df["type"] != "reliable").astype(int)
y = df["label"].values

### Simple model:

In [ ]:
#Simple model here (reusing from Part 2):
word_freq = pickle.load(open("word_freq.pkl", "rb"))

top_words = [w for w, c in word_freq.most_common(10000)]
vocab_lr = {word: i for i, word in enumerate(top_words)}

vectorizer_lr = CountVectorizer(vocabulary=vocab_lr)
X_lr = vectorizer_lr.transform(df["content"].fillna(""))

X_train_lr, X_temp_lr, y_train_lr, y_temp_lr = train_test_split(
    X_lr, y, test_size=0.2, random_state=69
)

X_val_lr, X_test_lr, y_val_lr, y_test_lr = train_test_split(
    X_temp_lr, y_temp_lr, test_size=0.5, random_state=69
)

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_lr, y_train_lr)

y_pred_lr = model_lr.predict(X_test_lr)
f1_lr = f1_score(y_test_lr, y_pred_lr)

print(f"Simple model F1: {f1_lr:.4f}")

Simple model F1: 0.9676


c:\Users\Nickl\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Complex model:

In [5]:
#Complex model here (Reusing from Part 3):
train_df_svm, temp_df_svm = train_test_split(
    df, test_size=0.2, random_state=69
)

val_df_svm, test_df_svm = train_test_split(
    temp_df_svm, test_size=0.5, random_state=69
)

y_train_svm = train_df_svm["label"].values
y_val_svm   = val_df_svm["label"].values
y_test_svm  = test_df_svm["label"].values

vectorizer_svm = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_svm = vectorizer_svm.fit_transform(train_df_svm["content"].fillna(""))
X_val_svm   = vectorizer_svm.transform(val_df_svm["content"].fillna(""))
X_test_svm  = vectorizer_svm.transform(test_df_svm["content"].fillna(""))

model_svm = LinearSVC(C=1.0)
model_svm.fit(X_train_svm, y_train_svm)

val_pred_svm = model_svm.predict(X_val_svm)
test_pred_svm = model_svm.predict(X_test_svm)

print(f"Advanced model validation F1: {f1_score(y_val_svm, val_pred_svm):.4f}")
print(f"Advanced model test F1: {f1_score(y_test_svm, test_pred_svm):.4f}")

Advanced model validation F1: 0.9768
Advanced model test F1: 0.9760


### Storing models:

In [ ]:
#Import packages:
import joblib

#Save it:
joblib.dump(vectorizer_lr, "vectorizer_lr.joblib")
joblib.dump(model_lr, "model_lr.joblib")

joblib.dump(vectorizer_svm, "vectorizer_svm.joblib")
joblib.dump(model_svm, "model_svm.joblib")

### If needing models later:

In [ ]:
vectorizer_lr = joblib.load("vectorizer_lr.joblib")
model_lr = joblib.load("model_lr.joblib")

vectorizer_svm = joblib.load("vectorizer_svm.joblib")
model_svm = joblib.load("model_svm.joblib")

### Task 2 - Do the same on LIAR dataset.

In [ ]:
#Load data:
liar_test = pd.read_csv("test.tsv", sep="\t", header=None)

In [7]:
#Extract relevant columns:
liar_test = liar_test[[1, 2]]
liar_test.columns = ["label", "content"]

In [8]:
#Convert labels to binary in the same spirit as previous task:
reliable = ["true", "mostly-true"]
fake = ["false", "pants-fire", "barely-true", "half-true"]
liar_test["label_bin"] = liar_test["label"].apply(
    lambda x: 0 if x in reliable else 1
)

In [9]:
#Vectorize, while NOT fitting again:
X_liar_lr = vectorizer_lr.transform(liar_test["content"].fillna(""))
X_liar_svm = vectorizer_svm.transform(liar_test["content"].fillna(""))

In [10]:
#Make prediction:
y_liar = liar_test["label_bin"].values

y_pred_lr_liar = model_lr.predict(X_liar_lr)
y_pred_svm_liar = model_svm.predict(X_liar_svm)

In [ ]:
#Perform evaluation:
f1_lr_liar = f1_score(y_liar, y_pred_lr_liar)
f1_svm_liar = f1_score(y_liar, y_pred_svm_liar)

print(f"LR on LIAR: {f1_lr_liar:.4f}")
print(f"SVM on LIAR: {f1_svm_liar:.4f}")
print(f"LR on LIAR ROC AUC: {roc_auc_score(y_liar, model_lr.predict_proba(X_liar_lr)[:,1]):.4f"}")
print(f"SVM on LIAR ROC AUC: {roc_auc_score(y_liar, model_svm.decision_function(X_liar_svm)):.4f"}")
print("LR on LIAR confusion matrix:")
print(confusion_matrix(y_liar, y_pred_lr_liar))
print("SVM on LIAR confusion matrix:")
print(confusion_matrix(_liar, y_pred_lr_liar))

print(classification_report(y_test_lr, y_pred_lr))

In [ ]:
#Improved and corrected evaluation:
y_pred_lr_liar = model_lr.predict(X_liar_lr)
y_pred_svm_liar = model_svm.predict(X_liar_svm)

#F1 scores:
print(f"LR on LIAR: {f1_score(y_liar, y_pred_lr_liar):.4f}")
print(f"SVM on LIAR: {f1_score(y_liar, y_pred_svm_liar):.4f}")

#Confusion matrices:
print("LR confusion matrix:\n", confusion_matrix(y_liar, y_pred_lr_liar))
print("SVM confusion matrix:\n", confusion_matrix(y_liar, y_pred_svm_liar))

#Classification reports (nice for interpretation):
print("LR report:\n", classification_report(y_liar, y_pred_lr_liar))
print("SVM report:\n", classification_report(y_liar, y_pred_svm_liar))

LR on LIAR: 0.7847
SVM on LIAR: 0.7790
LR confusion matrix:
 [[  0 449]
 [  0 818]]
SVM confusion matrix:
 [[ 10 439]
 [ 16 802]]
LR report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       449
           1       0.65      1.00      0.78       818

    accuracy                           0.65      1267
   macro avg       0.32      0.50      0.39      1267
weighted avg       0.42      0.65      0.51      1267

SVM report:
               precision    recall  f1-score   support

           0       0.38      0.02      0.04       449
           1       0.65      0.98      0.78       818

    accuracy                           0.64      1267
   macro avg       0.52      0.50      0.41      1267
weighted avg       0.55      0.64      0.52      1267



c:\Users\Nickl\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Nickl\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Nickl\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
